In [1]:
import pandas as pd
from datasets import load_dataset
from transformers import pipeline
import torch

d:\LPA_Part2\lpa2venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# jupyter nbconvert --to python rrl.ipynb

In [3]:
# 1. Initialize the new model pipeline
# This uses the specific model you found for segmentation
segmenter = pipeline(
    "text-classification", 
    model="Ansh-Singhal/inlegalbert-legalseg",
    device=0 if torch.cuda.is_available() else -1 # Uses your Y540's GPU if available
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 255.39it/s, Materializing param=classifier.weight]                                      


In [4]:
# 2. Load dataset in streaming mode
dataset = load_dataset("opennyaiorg/InJudgements_dataset", split="train", streaming=True)

In [5]:
def segment_text(text):
    # Transformer models have a limit (usually 512 tokens)
    # We split the first few paragraphs to extract facts/evidence
    paragraphs = text.split('\n')[:15] 
    
    facts, evidence = [], []
    
    # for para in paragraphs:
    #     if len(para.strip()) < 20: continue # Skip empty/short lines
        
    #     # Predict the role of the paragraph
    #     prediction = segmenter(para[:512])[0] # Truncate to avoid error
    #     label = prediction['label']
        
    #     if 'FACT' in label:
    #         facts.append(para.strip())
    #     elif 'EVIDENCE' in label:
    #         evidence.append(para.strip())

    for para in paragraphs:
        if len(para.strip()) < 20: continue
        
        prediction = segmenter(para[:512])[0]
        label = prediction['label']
        
        # DEBUG: See what the model is actually saying
        print(f"Detected Label: {label}") 
        
        # Use .upper() to be safe and check for common variations
        if 'FACT' in label.upper() or 'FAC' == label.upper():
            facts.append(para.strip())
        elif 'EVIDENCE' in label.upper() or 'EVI' in label.upper():
            evidence.append(para.strip())
            
    return " ".join(facts), " ".join(evidence)

In [6]:
# 3. Process the first dozen cases
results = []
print("Segmenting cases with InLegalBERT-LegalSeg...")

Segmenting cases with InLegalBERT-LegalSeg...


In [7]:
for i, record in enumerate(dataset):
    if i >= 12: break
    f, e = segment_text(record['Text'])
    results.append({
        "Case_ID": i,
        "Facts": f,
        "Evidence": e
    })

Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Reasoning
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Arguments of Petitioner
Detected Label: Arguments of Respondent
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Facts
Detected Label: Decision
Detected Label: Facts
Detected Label: Arguments of Resp

In [8]:
df = pd.DataFrame(results)
df.to_csv("mtech_bert_extraction.csv", index=False)
print("Done! Check 'mtech_bert_extraction.csv'.")

Done! Check 'mtech_bert_extraction.csv'.
